# AnyCount — Apple COCO Pilot Pack (Colab, sekali jalan)

Notebook ini membangun pack detektor `apple` dari **COCO 2017**, hanya mengunduh gambar
yang benar-benar berisi kelas `apple`, melatih YOLO11, lalu mengekspor **ONNX INT8**
yang kompatibel dengan `prototype/workers/onnx_worker.py`.

**Cara pakai**
1. `Runtime` → `Change runtime type` → **T4 GPU**.
2. `Runtime` → **Run all**.
3. Tunggu ±45–70 menit (tergantung `EPOCHS`); hasil akhirnya dua file yang otomatis
   terunduh: `apple-coco_v1_int8.onnx` dan `apple-coco_v1_bundle.zip`.
4. Kirim `.onnx` itu balik ke agent untuk didaftarkan sebagai pack dan diuji dengan GT manual.

Catatan jujur: label COCO tidak menandai semua instance pada tumpukan padat, jadi pack ini
bagus untuk apel "biasa" (foto pasar/dapur) tetapi belum tentu untuk tumpukan rapat.
Metrik yang dipakai di akhir adalah **count MAPE**, bukan hanya mAP.

## 1) Dependensi

In [ ]:
!pip -q install "ultralytics>=8.3" onnx onnxruntime pillow requests tqdm

import sys, torch, ultralytics
print("python:", sys.version.split()[0])
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("PERINGATAN: GPU tidak aktif. Runtime > Change runtime type > T4 GPU, lalu Run all lagi.")

## 2) Konfigurasi

In [ ]:
from pathlib import Path
import random

CLASS_NAME = "apple"            # kelas COCO target
MODEL_WEIGHTS = "yolo11s.pt"    # yolo11n.pt = cepat, yolo11m.pt = lebih akurat
EPOCHS = 80
IMG_SIZE = 640
BATCH = 16
MAX_TRAIN_IMAGES = 1200
MAX_VAL_IMAGES = 250
MIN_BOX_PX = 4                  # buang bbox yang lebih kecil dari ini
SEED = 7
PACK_SLUG = "apple-coco"

USE_DRIVE = False               # True untuk menyimpan hasil ke Google Drive
DRIVE_DIR = "/content/drive/MyDrive/anycount_packs"

WORK = Path("/content/anycount-coco")
COCO = WORK / "coco"
DS = WORK / "dataset"
RUNS = WORK / "runs"
WORK.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print("workdir:", WORK)

## 3) Unduh anotasi COCO 2017 (hanya file instances yang diekstrak)

In [ ]:
import shutil
import urllib.request, zipfile

ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANN_ZIP = COCO / "annotations_trainval2017.zip"
ANN_DIR = COCO / "annotations"
ANN_DIR.mkdir(parents=True, exist_ok=True)

if not ANN_ZIP.exists():
    print("mengunduh anotasi (~241 MB)...")
    urllib.request.urlretrieve(ANN_URL, ANN_ZIP)

# Zip menyimpan path "annotations/<file>"; ekstrak rata ke ANN_DIR.
needed = {"instances_train2017.json", "instances_val2017.json"}
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in z.namelist():
        base = Path(name).name
        target = ANN_DIR / base
        if base in needed and not target.exists():
            print("extract", base)
            with z.open(name) as fsrc, target.open("wb") as fdst:
                shutil.copyfileobj(fsrc, fdst, length=1 << 20)

found = sorted(p.name for p in ANN_DIR.glob("*.json"))
print(found)
if not needed.issubset(set(found)):
    raise SystemExit(f"anotasi tidak lengkap: {found}")

## 4) Filter gambar berisi `apple`, unduh paralel, konversi ke YOLO

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm.auto import tqdm

def load_split(split):
    path = ANN_DIR / f"instances_{split}2017.json"
    with path.open() as f:
        return json.load(f)

def target_category(data):
    matches = [c for c in data["categories"] if c["name"] == CLASS_NAME]
    if not matches:
        raise SystemExit(f"kelas {CLASS_NAME!r} tidak ada di COCO")
    return matches[0]["id"]

def collect(split, limit):
    data = load_split(split)
    cat_id = target_category(data)
    images = {im["id"]: im for im in data["images"]}
    boxes = {}
    for ann in data["annotations"]:
        if ann["category_id"] != cat_id or ann.get("iscrowd"):
            continue
        x, y, w, h = ann["bbox"]
        if w < MIN_BOX_PX or h < MIN_BOX_PX:
            continue
        boxes.setdefault(ann["image_id"], []).append((x, y, w, h))
    ids = sorted(boxes)
    if limit and len(ids) > limit:
        ids = random.sample(ids, limit)
    return [(images[i], boxes[i]) for i in ids]

train_items = collect("train", MAX_TRAIN_IMAGES)
val_items = collect("val", MAX_VAL_IMAGES)
print(f"kandidat train: {len(train_items)} | val: {len(val_items)}")

def download(item, split):
    im, _ = item
    dst = DS / "images" / split / im["file_name"]
    if dst.exists():
        return dst
    url = f"http://images.cocodataset.org/{split}2017/{im['file_name']}"
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            dst.write_bytes(r.content)
            return dst
        except Exception:
            if attempt == 2:
                raise
    return dst

def materialize(items, split):
    (DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / split).mkdir(parents=True, exist_ok=True)
    written = 0
    with ThreadPoolExecutor(max_workers=16) as pool:
        futs = {pool.submit(download, it, split): it for it in items}
        for fut in tqdm(as_completed(futs), total=len(futs), desc=f"download {split}"):
            im, boxes = futs[fut]
            fut.result()
            w, h = im["width"], im["height"]
            lines = [
                f"0 {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}"
                for x, y, bw, bh in boxes
            ]
            (DS / "labels" / split / f"{Path(im['file_name']).stem}.txt").write_text(
                "\n".join(lines) + "\n"
            )
            written += 1
    return written

n_train = materialize(train_items, "train")
n_val = materialize(val_items, "val")
print(f"train: {n_train} gambar | val: {n_val} gambar")

## 5) `data.yaml` + statistik dataset

In [ ]:
box_counts = [len(p.read_text().splitlines()) for p in (DS / "labels" / "train").glob("*.txt")]
total_boxes = sum(box_counts)
print("total bbox train:", total_boxes)
print("rata-rata bbox/gambar:", round(total_boxes / max(1, len(box_counts)), 2))
print("gambar tanpa bbox (harus 0):", sum(1 for c in box_counts if c == 0))

yaml_path = DS / "data.yaml"
yaml_path.write_text(
    f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n  0: {CLASS_NAME}\n"
)
print(yaml_path.read_text())

## 6) Cek visual 4 sampel (bbox dari label YOLO)

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

for img_path in sorted((DS / "images" / "train").glob("*.jpg"))[:4]:
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    w, h = im.size
    for line in (DS / "labels" / "train" / f"{img_path.stem}.txt").read_text().splitlines():
        _, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        draw.rectangle(
            [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2],
            outline=(255, 0, 0),
            width=3,
        )
    im.thumbnail((480, 480))
    display(im)

## 7) Training

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL_WEIGHTS)
model.train(
    data=str(DS / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=device,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=0.7,
    close_mosaic=15,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    fliplr=0.5,
    patience=25,
    workers=2,
    project=str(RUNS),
    name=f"{PACK_SLUG}_v1",
    exist_ok=True,
    seed=SEED,
    verbose=True,
)

best = RUNS / f"{PACK_SLUG}_v1" / "weights" / "best.pt"
print("best:", best, "| ada:", best.exists())

## 8) Validasi mAP + MAPE count

In [ ]:
metrics = model.val(data=str(DS / "data.yaml"), imgsz=IMG_SIZE, device=device)
print(f"mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")

val_images = sorted((DS / "images" / "val").glob("*.jpg"))[:30]
errs = []
for p in val_images:
    truth = len((DS / "labels" / "val" / f"{p.stem}.txt").read_text().splitlines())
    if truth == 0:
        continue
    pred = model.predict(str(p), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
    count = 0 if pred.boxes is None else len(pred.boxes)
    errs.append(abs(count - truth) / truth)
print(f"count MAPE ({len(errs)} gambar val, conf 0.25): {100 * sum(errs) / max(1, len(errs)):.1f}%")

## 9) Ekspor ONNX + kuantisasi INT8

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

best = RUNS / f"{PACK_SLUG}_v1" / "weights" / "best.pt"
model = YOLO(str(best))
onnx_fp32 = Path(model.export(format="onnx", imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True))
int8_path = WORK / f"{PACK_SLUG}_v1_int8.onnx"
quantize_dynamic(str(onnx_fp32), str(int8_path), weight_type=QuantType.QInt8)

print(f"FP32: {onnx_fp32.stat().st_size / 1e6:.2f} MB")
print(f"INT8: {int8_path.stat().st_size / 1e6:.2f} MB -> {int8_path}")

## 10) Sanity check ONNX INT8

In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image

sess = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
print("input:", inp.name, inp.shape, inp.type)

img = Image.open(val_images[0]).convert("RGB")
w, h = img.size
scale = min(IMG_SIZE / w, IMG_SIZE / h)
nw, nh = round(w * scale), round(h * scale)
canvas = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (114, 114, 114))
canvas.paste(img.resize((nw, nh), Image.BILINEAR), ((IMG_SIZE - nw) // 2, (IMG_SIZE - nh) // 2))
arr = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0

out = sess.run(None, {inp.name: arr})[0]
print("output shape:", out.shape)
if out.ndim == 3 and out.shape[1] >= 5 and out.shape[2] > out.shape[1]:
    scores = out[0, 4:, :].max(axis=0)   # (1, 4+nc, anchors)
elif out.ndim == 3:
    scores = out[0, :, 4:].max(axis=1)   # (1, anchors, 4+nc)
else:
    scores = np.array([float("nan")])
print("max score:", float(np.nanmax(scores)))

## 11) Kemas hasil + unduh

In [ ]:
import json, shutil, zipfile

summary = {
    "class": CLASS_NAME,
    "model": MODEL_WEIGHTS,
    "epochs": EPOCHS,
    "imgsz": IMG_SIZE,
    "train_images": n_train,
    "val_images": n_val,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "count_mape_pct": round(100 * sum(errs) / max(1, len(errs)), 2),
    "onnx_int8_mb": round(int8_path.stat().st_size / 1e6, 2),
}
(WORK / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

artifact = WORK / f"{PACK_SLUG}_v1_bundle.zip"
run_dir = RUNS / f"{PACK_SLUG}_v1"
with zipfile.ZipFile(artifact, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(int8_path, int8_path.name)
    z.write(WORK / "summary.json", "summary.json")
    z.write(run_dir / "results.csv", "results.csv")
    z.write(run_dir / "weights" / "best.pt", "best.pt")
    for p in sorted(run_dir.glob("*.png")):
        z.write(p, f"plots/{p.name}")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy(int8_path, Path(DRIVE_DIR) / int8_path.name)
    shutil.copy(artifact, Path(DRIVE_DIR) / artifact.name)
    print("tersimpan di Drive:", DRIVE_DIR)

try:
    from google.colab import files
    files.download(str(int8_path))
    files.download(str(artifact))
except Exception as exc:  # bukan Colab: file tetap ada di WORK
    print("auto-download tidak tersedia:", exc)

print("Selesai. Kirim file .onnx ke agent untuk didaftarkan sebagai pack.")
print("Lokasi file:", int8_path)